In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from enum import IntEnum
from functools import lru_cache

In [ ]:
df = pd.read_csv('../results_new.csv', comment='#')
print(df.shape)
df.head(3)

In [ ]:
df['delta1'] = df['delta1'].round(6)
df['delta2'] = df['delta2'].round(6)
df['eps'] = df['eps'].round(6)

In [ ]:
COUPLING_NAMES = ['Inertial', 'InertialNorm', 'Dissipative', 'DissipativeNorm']
epsilons = sorted(df['eps'].unique())


class CouplingType(IntEnum):
    Inertial = 0
    InertialNorm = 1
    Dissipative = 2
    DissipativeNorm = 3


# Порог наклона разности фаз: |slope| < θ ⇒ пара синхронизирована.
THETA = {ct: 0.025 for ct in CouplingType}

REGIME_LABELS = ['нет синхр.', '0↔1', '0↔2', '0↔1, 0↔2', '1↔2', '0↔1, 1↔2', '0↔2, 1↔2', 'полная']
REGIME_COLORS = ['#d3d3d3', '#4477aa', '#66ccee', '#228833', '#ccbb44', '#ee6677', '#aa3377', '#222222']

print('epsilons:', epsilons)
print('coupling types:', sorted(df['coupling_type'].unique()))
print('runs per point:', df.groupby(['delta1', 'delta2', 'eps', 'coupling_type']).size().unique())

In [ ]:
def codes_for(sub):
    """Битовая маска синхронизации: bit0 = 0↔1, bit1 = 0↔2, bit2 = 1↔2."""
    theta = sub['coupling_type'].map(THETA)
    return ((sub['s01'] < theta).astype(int)
            + (sub['s02'] < theta).astype(int) * 2
            + (sub['s12'] < theta).astype(int) * 4)


@lru_cache(maxsize=None)
def sub_with_code(eps):
    sub = df[df['eps'] == eps].copy()
    sub['code'] = codes_for(sub)
    return sub


def discrete_colorscale(colors):
    """Ступенчатая plotly colorscale для дискретных кодов 0..len(colors)-1."""
    n = len(colors)
    return [pt for i, c in enumerate(colors) for pt in ([i / n, c], [(i + 1) / n, c])]


REGIME_SCALE = discrete_colorscale(REGIME_COLORS)


def _eps_step(name):
    return dict(method='animate', label=name,
                args=[[name], dict(mode='immediate', frame=dict(duration=0, redraw=True),
                                   transition=dict(duration=0))])


def add_eps_slider(fig, frames, prefix='ε = ', pad_t=60):
    """Нативный plotly-слайдер по ε. frames: list[go.Frame] (name=str(eps)). Без кнопок play/stop."""
    fig.frames = list(frames)
    fig.update_layout(sliders=[dict(active=0, pad={'t': pad_t},
                                    currentvalue={'prefix': prefix},
                                    steps=[_eps_step(f.name) for f in fig.frames])])
    return fig


def add_eps_coupling_sliders(fig, frames, group_size, coupling_names):
    """Два независимых нативных слайдера без play/stop:
       • ε — кадры (go.Frame) обновляют данные z всех трасс (видимость не трогают);
       • тип связи — переключает видимость групп трасс (по group_size трасс на тип).
       Трассы фигуры идут связь-мажорным порядком; кадры НЕ задают `visible`."""
    fig.frames = list(frames)
    total = len(coupling_names) * group_size
    coupling_steps = [dict(method='restyle', label=name,
                           args=[{'visible': [ci * group_size <= i < (ci + 1) * group_size
                                              for i in range(total)]}])
                      for ci, name in enumerate(coupling_names)]
    fig.update_layout(sliders=[
        dict(active=0, yanchor='top', y=0, pad={'t': 40},
             currentvalue={'prefix': 'ε = '}, steps=[_eps_step(f.name) for f in fig.frames]),
        dict(active=0, yanchor='top', y=0, pad={'t': 110},
             currentvalue={'prefix': 'тип связи: '}, steps=coupling_steps),
    ])
    return fig

## Подбор порога θ

### Распределение наклонов разности фаз

In [ ]:
# KDE плотности |slope| по всем парам осцилляторов. Фасет — тип связи, слайдер — ε.
def kde_long():
    x_grid = np.linspace(0, 0.2, 400)
    out = []
    for e in epsilons:
        sub = df[df['eps'] == e]
        s = np.abs(np.concatenate([sub['s01'].values, sub['s02'].values, sub['s12'].values]))
        ct = np.tile(sub['coupling_type'].values, 3)
        for c in range(4):
            y = gaussian_kde(s[ct == c], bw_method=0.03)(x_grid)
            out.append(pd.DataFrame({'eps': e, 'coupling': COUPLING_NAMES[c],
                                     '|slope|': x_grid, 'density': y}))
    return pd.concat(out, ignore_index=True)


kdf = kde_long()
fig = px.line(kdf, x='|slope|', y='density',
              facet_col='coupling', facet_col_wrap=2,
              animation_frame='eps', log_y=True, height=800, width=800,
              title='Плотность |slope| (KDE)')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
# Фикс. диапазон по всем кадрам, чтобы кривые не уходили за рабочую зону.
dens = kdf['density']
fig.update_yaxes(range=[np.log10(dens[dens > 0].min()), np.log10(dens.max() * 1.2)])
fig.layout.updatemenus = []   # убрать кнопки play/stop, оставить только слайдер
fig.show()

In [ ]:
# Выбранный порог θ для каждого типа связи.
pd.Series({CouplingType(ct).name: THETA[CouplingType(ct)] for ct in range(4)}, name='θ').to_frame()

## Карты режимов синхронизации

In [ ]:
# Доминирующий код режима и флаг мультистабильности в каждой точке (δ₁, δ₂).
def regime_agg(eps):
    sub = sub_with_code(eps)
    counts = sub.groupby(['delta1', 'delta2', 'coupling_type', 'code']).size()
    g = counts.groupby(level=[0, 1, 2])
    dominant = g.idxmax().map(lambda x: x[3]).rename('dominant_code')
    multi = (g.count() > 1).rename('multi_regime')
    return pd.concat([dominant, multi], axis=1).reset_index()


all_agg = {e: regime_agg(e) for e in epsilons}

In [ ]:
# Карта режимов: цвет = доминирующий код, контур = зона мультистабильности.
# Слайдеры — ε и тип связи.

_LBL_ARR = np.array(REGIME_LABELS)


def regime_heatmap(dom):
    return go.Heatmap(
        z=dom.values,
        x=dom.columns.tolist(),
        y=dom.index.tolist(),
        customdata=_LBL_ARR[dom.values.astype(int)],
        colorscale=REGIME_SCALE,
        zmin=-0.5,
        zmax=7.5,
        showscale=False,
        hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>%{customdata}<extra></extra>'
    )


def regime_border(mul):
    return go.Contour(
        z=mul.values,
        x=mul.columns.tolist(),
        y=mul.index.tolist(),
        contours=dict(start=0.5, end=0.5, size=1, coloring='lines'),
        colorscale=[[0, '#444'], [1, '#444']],
        line=dict(width=1),
        showscale=False,
        hoverinfo='skip'
    )


def regime_traces(eps):
    """Связь-мажорный порядок: на тип связи по 2 трассы
    (режим + контур мультистабильности).
    """
    a = all_agg[eps]
    traces = []

    for ct in range(4):
        d = a[a['coupling_type'] == ct]

        dom = d.pivot(
            index='delta1',
            columns='delta2',
            values='dominant_code'
        )

        mul = d.pivot(
            index='delta1',
            columns='delta2',
            values='multi_regime'
        ).astype(float)

        traces.append(regime_heatmap(dom))
        traces.append(regime_border(mul))

    return traces


fig = go.Figure()

# Начальное ε: добавляем все 4 типа связи,
# но видимым оставляем только первый тип связи.
for idx, tr in enumerate(regime_traces(epsilons[0])):
    tr.visible = (idx // 2 == 0)
    fig.add_trace(tr)

frames = [
    go.Frame(
        name=str(e),
        data=regime_traces(e)
    )
    for e in epsilons
]

fig.update_layout(
    height=800,
    width=800,
    autosize=True,
    plot_bgcolor='white',
    margin=dict(b=140),
    xaxis_title='δ₂',
    yaxis_title='δ₁',
    title='Режимы синхронизации (контур = мультистабильность)'
)

add_eps_coupling_sliders(
    fig,
    frames,
    group_size=2,
    coupling_names=COUPLING_NAMES
)

fig.show()

## Целевые функции

In [ ]:
# Средние целевые функции L, A, P. Столбцы — функция, слайдеры — ε и тип связи.
# У каждой панели своя авто-шкала цвета (значение в подсказке при наведении).
_GOALS = ['L', 'A', 'P']


def goal_heatmap(piv):
    return go.Heatmap(z=piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
                      colorscale='YlOrRd', showscale=False,
                      hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>%{z:.3f}<extra></extra>')


def goal_traces(eps):
    """Связь-мажорный порядок: c0:(L,A,P), c1:(L,A,P), … (3 трассы на тип связи)."""
    agg = (df[df['eps'] == eps]
           .groupby(['delta1', 'delta2', 'coupling_type'])[_GOALS].mean().reset_index())
    return [goal_heatmap(agg[agg['coupling_type'] == ct].pivot(
                index='delta1', columns='delta2', values=g))
            for ct in range(4) for g in _GOALS]


fig = make_subplots(rows=1, cols=3, subplot_titles=_GOALS, horizontal_spacing=0.07)
for idx, tr in enumerate(goal_traces(epsilons[0])):
    tr.visible = (idx // 3 == 0)        # изначально видна только первая связь
    fig.add_trace(tr, row=1, col=idx % 3 + 1)

frames = [go.Frame(name=str(e), data=goal_traces(e)) for e in epsilons]
fig.update_layout(height=600, width=1200, margin=dict(b=140), title='Целевые функции L, A, P')
add_eps_coupling_sliders(fig, frames, group_size=3, coupling_names=COUPLING_NAMES)
fig.show()

In [ ]:
# Корреляции L, A, P (нижний треугольник). Фасет — тип связи, слайдер — ε.
_LAB = ['L', 'A', 'P']
_TRIL = np.tril(np.ones((3, 3)), -1).astype(bool)


def corr_panels(eps):
    agg = df[df['eps'] == eps].groupby(['delta1', 'delta2', 'coupling_type'])[_LAB].mean()
    return [np.where(_TRIL, agg.xs(ct, level='coupling_type').corr().values, np.nan)
            for ct in range(4)]


def corr_heatmap(z, showscale):
    text = [[f'{z[i, j]:.2f}' if not np.isnan(z[i, j]) else '' for j in range(3)] for i in range(3)]
    return go.Heatmap(z=z, x=_LAB, y=_LAB, text=text, texttemplate='%{text}',
                      colorscale='RdBu', zmin=-1, zmax=1, showscale=showscale,
                      hovertemplate='%{y}/%{x}: %{z:.3f}<extra></extra>')


fig = make_subplots(rows=1, cols=4, subplot_titles=COUPLING_NAMES, horizontal_spacing=0.08)
for i, z in enumerate(corr_panels(epsilons[0])):
    fig.add_trace(corr_heatmap(z, showscale=(i == 3)), row=1, col=i + 1)

frames = [go.Frame(name=str(e), data=[corr_heatmap(z, showscale=(i == 3))
                                      for i, z in enumerate(corr_panels(e))])
          for e in epsilons]

fig.update_layout(height=600, title='Корреляции L, A, P (нижний треугольник)')
add_eps_slider(fig, frames)
fig.show()

In [ ]:
# Сводка попарных корреляций по всем ε и типам связи.
rows = []
for e in epsilons:
    agg = df[df['eps'] == e].groupby(['delta1', 'delta2', 'coupling_type'])[['L', 'A', 'P']].mean()
    for ct in range(4):
        c = agg.xs(ct, level='coupling_type').corr()
        rows.append({'ε': e, 'coupling': COUPLING_NAMES[ct],
                     'ρ(L,A)': c.loc['L', 'A'], 'ρ(L,P)': c.loc['L', 'P'], 'ρ(A,P)': c.loc['A', 'P']})
pd.DataFrame(rows).round(3)

## Синфазность

Карты режимов отвечают на вопрос «где синхронно», но не «где синфазно». Здесь поверх
приглушённой карты режимов (фон) выводится $P$ только в зоне полной синхронизации (код 7).

In [ ]:
# Синфазность: ⟨P⟩ в точках, где зона полной синхронизации (код 7) преобладает (>50% НУ).
# Фон — приглушённая карта режимов, сверху — ⟨P⟩. Слайдеры — ε и тип связи.
def inphase_agg(eps):
    sub = sub_with_code(eps)
    keys = ['delta1', 'delta2', 'coupling_type']
    counts = sub.groupby(keys + ['code']).size().reset_index(name='n')
    dom = (counts.sort_values('n').drop_duplicates(keys, keep='last')
           .rename(columns={'code': 'dominant_code'})[keys + ['dominant_code']])
    frac = (sub.assign(full=sub['code'] == 7).groupby(keys)['full'].mean()
            .rename('full_frac').reset_index())
    psync = sub[sub['code'] == 7].groupby(keys)['P'].mean().rename('P_sync').reset_index()
    agg = dom.merge(frac, on=keys).merge(psync, on=keys, how='left')
    agg.loc[agg['full_frac'] < 0.5, 'P_sync'] = np.nan
    return agg


def inphase_traces(eps):
    """Связь-мажорный порядок: на тип связи по 2 трассы (фон-режим + ⟨P⟩)."""
    agg = inphase_agg(eps)
    traces = []
    for ct in range(4):
        d = agg[agg['coupling_type'] == ct]
        dom = d.pivot(index='delta1', columns='delta2', values='dominant_code')
        p = d.pivot(index='delta1', columns='delta2', values='P_sync')
        traces.append(go.Heatmap(z=dom.values, x=dom.columns.tolist(), y=dom.index.tolist(),
                                 colorscale=REGIME_SCALE, zmin=-0.5, zmax=7.5,
                                 opacity=0.35, showscale=False, hoverinfo='skip'))
        traces.append(go.Heatmap(z=p.values, x=p.columns.tolist(), y=p.index.tolist(),
                                 colorscale='Viridis', zmin=1 / 3, zmax=1.0,
                                 showscale=True, colorbar=dict(title='⟨P⟩'),
                                 hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>⟨P⟩=%{z:.3f}<extra></extra>'))
    return traces


fig = go.Figure()
for idx, tr in enumerate(inphase_traces(epsilons[0])):
    tr.visible = (idx // 2 == 0)        # изначально видна только первая связь
    fig.add_trace(tr)

frames = [go.Frame(name=str(e), data=inphase_traces(e)) for e in epsilons]
fig.update_layout(height=800, width=800, margin=dict(b=140), xaxis_title='δ₂', yaxis_title='δ₁',
                  title='⟨P⟩ в зоне полной синхронизации (фон = режим)')
add_eps_coupling_sliders(fig, frames, group_size=2, coupling_names=COUPLING_NAMES)
fig.show()

## Исследование $\Delta L$, $\Delta A$, $\Delta P$ как критериев мультистабильности

In [ ]:
# Распределение разбросов ΔL, ΔA, ΔP в моно- и мультистабильных точках.
# Строки — функция, столбцы — тип связи, цвет — мультистабильность, слайдер — ε.
_SPREADS = [('L', 'ΔL'), ('A', 'ΔA'), ('P', 'ΔP')]
_REG = [(False, 'один режим', 'steelblue'), (True, 'мультистаб.', 'tomato')]


def hist_agg(eps):
    return sub_with_code(eps).groupby(['delta1', 'delta2', 'coupling_type']).agg(
        multi=('code', lambda x: x.nunique() > 1),
        L=('L', lambda x: x.max() - x.min()),
        A=('A', lambda x: x.max() - x.min()),
        P=('P', lambda x: x.max() - x.min()),
    ).reset_index()


def hist_traces(agg):
    """24 трассы в фикс. порядке (ΔL,ΔA,ΔP) × связь × режим — для совместимости кадров."""
    traces = []
    for si, (col, _) in enumerate(_SPREADS):
        for ct in range(4):
            d = agg[agg['coupling_type'] == ct]
            for is_multi, name, color in _REG:
                traces.append(go.Histogram(
                    x=d.loc[d['multi'] == is_multi, col], nbinsx=50,
                    marker_color=color, opacity=0.6, name=name, legendgroup=name,
                    showlegend=(si == 0 and ct == 0)))
    return traces


fig = make_subplots(rows=3, cols=4, row_titles=[lbl for _, lbl in _SPREADS],
                    column_titles=COUPLING_NAMES, horizontal_spacing=0.05, vertical_spacing=0.07)
it = iter(hist_traces(hist_agg(epsilons[0])))
for si in range(3):
    for ct in range(4):
        for _ in _REG:
            fig.add_trace(next(it), row=si + 1, col=ct + 1)
fig.update_yaxes(type='log')

frames = [go.Frame(name=str(e), data=hist_traces(hist_agg(e))) for e in epsilons]
fig.update_layout(height=900, barmode='overlay', title='Разброс L, A, P vs мультистабильность')
add_eps_slider(fig, frames)
fig.show()

In [ ]:
# Контрпримеры для типа связи Inertial: мультистаб. при малом ΔL и моно при большом ΔL.
def counterexamples(eps, ct=0):
    agg = sub_with_code(eps).groupby(['delta1', 'delta2', 'coupling_type']).agg(
        multi=('code', lambda x: x.nunique() > 1),
        dL=('L', lambda x: x.max() - x.min()),
        codes=('code', list), L=('L', list), A=('A', list), P=('P', list),
    ).reset_index()
    d = agg[agg['coupling_type'] == ct]
    return (d[d['multi'] & (d['dL'] < 0.1)].head(3),
            d[~d['multi'] & (d['dL'] > 0.5)].head(3))


def show_counterexamples(eps, ct=0):
    ms, ml = counterexamples(eps, ct)
    print(f'=== ε={eps}, {COUPLING_NAMES[ct]} ===')
    print('Мультистаб. с малым ΔL (<0.1):')
    for _, r in ms.iterrows():
        print(f'  δ1={r.delta1:.2f} δ2={r.delta2:.2f} ΔL={r.dL:.4f} коды={r.codes}')
    print('Один режим с большим ΔL (>0.5):')
    for _, r in ml.iterrows():
        print(f'  δ1={r.delta1:.2f} δ2={r.delta2:.2f} ΔL={r.dL:.4f} L={[round(x, 3) for x in r.L]}')


show_counterexamples(epsilons[2])

### Проверка точки с маленьким $\Delta L$, но мультистабильностью

In [ ]:
# Все НУ в первой контрпример-точке: мультистаб. при малом ΔL.
def show_runs_at(eps, kind, ct=0):
    ms, ml = counterexamples(eps, ct)
    sel = ms if kind == 'multi_small' else ml
    if len(sel) == 0:
        print('нет таких точек в данных'); return
    r0 = sel.iloc[0]
    pt = sub_with_code(eps)
    pt = pt[(pt['coupling_type'] == ct)
            & np.isclose(pt['delta1'], r0.delta1) & np.isclose(pt['delta2'], r0.delta2)]
    print(f'ε={eps} δ1={r0.delta1:.2f} δ2={r0.delta2:.2f} ({COUPLING_NAMES[ct]}), ΔL={r0.dL:.4f}')
    print(pt[['code', 'L', 'x0', 'y0', 'x1', 'y1', 'x2', 'y2']].round(3).to_string(index=False))


show_runs_at(epsilons[2], 'multi_small')

### Проверка точки с высоким $\Delta L$, но однорежимностью

In [ ]:
# Все НУ в первой контрпример-точке: один режим при большом ΔL.
show_runs_at(epsilons[2], 'mono_large')

In [ ]:
# C++ vdp_sim: осцилляции в установившемся режиме (окно после переходного процесса).
t_trans, window = 600.0, 40.0
runs = [
    {'title': 'δ₁=-0.20 δ₂=0.12 — код 2 (0↔2), L≈0.834', 'path': '../simulation/demo/small_l_code2.csv'},
    {'title': 'δ₁=-0.20 δ₂=0.12 — код 0 (нет синхр.), L≈0.838', 'path': '../simulation/demo/small_l_code0.csv'},
    {'title': 'δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈2.1', 'path': '../simulation/demo/large_dl_high_l.csv'},
    {'title': 'δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈0.235', 'path': '../simulation/demo/large_dl_low_l.csv'},
]
colors = {'x0': '#e41a1c', 'x1': '#4daf4a', 'x2': '#377eb8'}

fig = make_subplots(rows=len(runs), cols=1, shared_xaxes=True,
                    subplot_titles=[r['title'] for r in runs])
for ri, run in enumerate(runs):
    dfr = pd.read_csv(run['path'], comment='#')
    steady = dfr[(dfr['t'] >= t_trans) & (dfr['t'] < t_trans + window)]
    for col, c in colors.items():
        fig.add_trace(go.Scatter(x=steady['t'], y=steady[col], mode='lines',
                                 line=dict(color=c, width=1), name=col,
                                 legendgroup=col, showlegend=(ri == 0)),
                      row=ri + 1, col=1)

fig.update_xaxes(title_text='t', row=len(runs), col=1)
fig.update_layout(height=1000, title='C++ vdp_sim — осцилляции в установившемся режиме')
fig.show()